# Order-four cocycles · 逐项数学验证

选择 case 后，报告展示每个 contribution 和 exact identity。β₃ 的 combinatorial order 恰为四的证明不在本项目范围内。先从本目录启动 Jupyter，并安装 `pip install ".[notebook]"`。

Read the mathematical conventions in `docs/MATHEMATICS.md`. Every displayed identity is derived from oriented smoothing and linking numbers, not from numerical fitting.

Revision 1.1.0：实线核心弧／虚线补弧和 B 编号已统一；先看 `reports/zh/reading-guide.html`。残差增量只显示所选 β；证书列出逐系数比较。

Version 1.2.0：Language 下拉框选择报告语言。英文版与中文版的内容逐段对应；灰色箭头头部已加大，并与端点圆点分开。

In [ ]:
from pathlib import Path
import json, html
from IPython.display import display, HTML
import ipywidgets as widgets
from cocycle import reports, loops
from cocycle.verification import replay
bundle=json.loads(Path("results/local-certificates.json").read_text())
assert replay(bundle)
print("All supplied classical identity certificates replayed exactly.")
def show_document(document):
    import re
    document = re.sub(r'<nav class="jump-links">.*?</nav>', "", document, count=1)
    display(HTML('<iframe style="width:100%;height:850px;border:1px solid #ddd" srcdoc="'+html.escape(document, quote=True)+'"></iframe>'))

language = widgets.Dropdown(options=[("English", "en"), ("中文", "zh")], value="zh", description="Language")
display(language)


## 局部方程 / Local equations

Tetrahedron: case `1` to `24`; cube: `template,rotation` (0–47,0–2); commute: `u,v,a` (0–47,0–47,0–9). Tetrahedron/cube 展示全部 outside-arrow 情况，包括非零 virtual 残差与消去它的 classical identity。

In [ ]:
family=widgets.Dropdown(options=["tetra","cube","commute"],description="Equation")
case=widgets.Text(value="1",description="Case")
beta=widgets.Dropdown(options=["beta1","beta2","beta3"],description="Cocycle")
button=widgets.Button(description="Show all contributions",layout=widgets.Layout(width="240px"))
out=widgets.Output()
def local_clicked(_):
    with out:
        out.clear_output(wait=True)
        parts=[int(x) for x in case.value.split(",")]
        if family.value=="commute": doc=reports.commutation_case(*parts,name=beta.value,lang=language.value)
        else: doc=reports.local_case(bundle,family.value,parts[0] if family.value=="tetra" else parts,beta.value,lang=language.value)
        show_document(doc)
button.on_click(local_clicked)
display(widgets.VBox([family,case,beta,button,out]))


## Loops

`half-rolling` 只适用于 `4_1_periodic`、`6_1_periodic` 等实际二周期零 framing 图。Rolling 的 raw notebook 方向与 main 相反，报告显式展示这个路径符号。

In [ ]:
kind=widgets.Dropdown(options=["rotation","rolling","half-rolling","bracket","half-bracket","push"],description="Loop")
knot=widgets.Dropdown(options=list(loops.knots()),value="4_1",description="Knot")
other=widgets.Dropdown(options=[None,"curl+","2curls+"]+list(loops.knots()),description="Second knot")
loopbutton=widgets.Button(description="Compute contributions",layout=widgets.Layout(width="240px"))
loopout=widgets.Output()
def loop_clicked(_):
    with loopout:
        loopout.clear_output(wait=True)
        result=loops.calculate(kind.value,knot.value,other.value)
        show_document(reports.loop_case(result,lang=language.value))
loopbutton.on_click(loop_clicked)
display(widgets.VBox([kind,knot,other,loopbutton,loopout]))


## 完整重算 / Full recomputation

下面单元格会重新穷举并计算全部局部方程和 loop examples，结果写入新目录。

In [ ]:
# Uncomment to recompute everything from the bundled source arrays.
# from cocycle.verification import run
# run("rerun-results")
# loops.verify("rerun-results")
